# GovernedRAG Retrieval Layer Experiments

## Objective

Design, evaluate, and optimize the retrieval layer of our existing RAG system.

We will establish the current baseline first and then experiment with:

- Chunking strategy
- Embedding model
- ANN/index configuration
- Retrieval `k`
- Query rewriting
- Dense vs hybrid retrieval
- Reranking

### Evaluation Metrics

- Recall@k
- Precision@k
- MRR
- nDCG
- Keyword Coverage
- Retrieval Latency

### Experiment Principle

Change one major retrieval component at a time,
evaluate it against the same dataset,
and keep changes only when the evidence supports them.

In [ ]:
# Imports 
from pathlib import Path
import sys


In [8]:


PROJECT_ROOT = Path.cwd()

sys.path.append(str(PROJECT_ROOT))

print("Project root:", PROJECT_ROOT)
print("Exists:", PROJECT_ROOT.exists())

Project root: /Users/tarkshya/Work/LLM Engineering/GovernedRAG
Exists: True


In [9]:
# Existing RAG project
WORKSPACE_ROOT = PROJECT_ROOT.parent
WEEK5_RAG = WORKSPACE_ROOT / "Week5_RAG"

sys.path.append(str(WEEK5_RAG))

print("Week5_RAG:", WEEK5_RAG)
print("Exists:", WEEK5_RAG.exists())

Week5_RAG: /Users/tarkshya/Work/LLM Engineering/Week5_RAG
Exists: True


In [ ]:
import implementation.answer as answer_module

print(answer_module.__file__)
print(dir(answer_module))



/Users/tarkshya/Work/LLM Engineering/Week5_RAG/implementation/answer.py
['ChatOpenAI', 'Chroma', 'DB_NAME', 'Document', 'HuggingFaceEmbeddings', 'HumanMessage', 'MODEL', 'OpenAIEmbeddings', 'Path', 'RETRIEVAL_K', 'SYSTEM_PROMPT', 'SystemMessage', '__builtins__', '__cached__', '__doc__', '__file__', '__loader__', '__name__', '__package__', '__spec__', 'answer_question', 'combined_question', 'convert_to_messages', 'embeddings', 'fetch_context', 'llm', 'load_dotenv', 'retriever', 'vectorstore']


In [15]:
matches = list(WEEK5_RAG.rglob("proAnswer.py"))

for path in matches:
    print(path)

/Users/tarkshya/Work/LLM Engineering/Week5_RAG/pro_implementation/proAnswer.py


In [16]:
from pro_implementation.proAnswer import (
    fetch_context,
    fetch_context_unranked,
    rewrite_query,
    rerank,
)

print("Pro retrieval pipeline loaded successfully.")

Pro retrieval pipeline loaded successfully.


In [17]:
import pro_implementation.proAnswer as baseline

print("Embedding model:", baseline.embedding_model)
print("Retrieval K:", baseline.RETRIEVAL_K)
print("Final K:", baseline.FINAL_K)
print("Database:", baseline.DB_NAME)
print("Collection:", baseline.collection.name)

Embedding model: text-embedding-3-large
Retrieval K: 20
Final K: 10
Database: /Users/tarkshya/Work/LLM Engineering/Week5_RAG/vector_db
Collection: docs


In [18]:
collection = baseline.collection

print("Total vectors:", collection.count())

sample = collection.get(
    limit=2,
    include=["documents", "metadatas"]
)

print("\n--- Sample document ---")
print(sample["documents"][0][:500])

print("\n--- Sample metadata ---")
print(sample["metadatas"][0])

Total vectors: 579

--- Sample document ---
Product Overview and Summary

Rellm is a cutting-edge AI-powered enterprise reinsurance platform developed by Insurellm, aimed at transforming risk management, decision-making, and operational efficiency in the reinsurance industry.

# Product Summary

# Rellm: AI-Powered Enterprise Reinsurance Solution

## Summary

Rellm is an innovative enterprise reinsurance product developed by Insurellm, designed to transform the way reinsurance companies operate. Harnessing the power of artificial intellig

--- Sample metadata ---
{'type': 'products', 'source': '/Users/tarkshya/Work/LLM Engineering/Week5_RAG/knowledge-base/products/Rellm.md'}


In [29]:
evaluation_files = list(WEEK5_RAG.rglob("*"))

for path in evaluation_files:
    if path.is_file() and "eval" in path.name.lower():
        print(path)

/Users/tarkshya/Work/LLM Engineering/Week5_RAG/evaluator.py
/Users/tarkshya/Work/LLM Engineering/Week5_RAG/evaluation/eval.py
/Users/tarkshya/Work/LLM Engineering/Week5_RAG/evaluation/__pycache__/eval.cpython-311.pyc


In [30]:
from evaluation.eval import evaluate_all_retrieval, evaluate_all_answers


In [31]:
import evaluation.eval as eval_module

print("Evaluation module:", eval_module.__file__)
print("Has evaluate_all_retrieval:",
      hasattr(eval_module, "evaluate_all_retrieval"))
print("Has evaluate_all_answers:",
      hasattr(eval_module, "evaluate_all_answers"))

Evaluation module: /Users/tarkshya/Work/LLM Engineering/Week5_RAG/evaluation/eval.py
Has evaluate_all_retrieval: True
Has evaluate_all_answers: True


In [32]:
print([
    name for name in dir(eval_module)
    if not name.startswith("_")
])

['AnswerEval', 'BaseModel', 'Field', 'MODEL', 'RetrievalEval', 'TestQuestion', 'answer_question', 'calculate_dcg', 'calculate_mrr', 'calculate_ndcg', 'completion', 'db_name', 'evaluate_all_answers', 'evaluate_all_retrieval', 'evaluate_answer', 'evaluate_retrieval', 'fetch_context', 'load_dotenv', 'load_tests', 'main', 'math', 'run_cli_evaluation', 'sys']


In [33]:
import inspect

print("fetch_context module:", eval_module.fetch_context.__module__)
print()
print(inspect.getsource(eval_module.fetch_context))

fetch_context module: implementation.answer

def fetch_context(question: str) -> list[Document]:
    """
    Retrieve relevant context documents for a question.
    """
    return retriever.invoke(question, k=RETRIEVAL_K)



In [34]:
print("answer_question module:", eval_module.answer_question.__module__)

answer_question module: implementation.answer


In [35]:
import inspect

source = inspect.getsource(eval_module)

print(source[:3000])

import sys
import math
from pydantic import BaseModel, Field
from litellm import completion
from dotenv import load_dotenv

from evaluation.test import TestQuestion, load_tests
from pro_implementation.proAnswer import answer_question, fetch_context


load_dotenv(override=True)

MODEL = "gpt-4.1-nano"
db_name = "vector_db"


class RetrievalEval(BaseModel):
    """Evaluation metrics for retrieval performance."""

    mrr: float = Field(description="Mean Reciprocal Rank - average across all keywords")
    ndcg: float = Field(description="Normalized Discounted Cumulative Gain (binary relevance)")
    keywords_found: int = Field(description="Number of keywords found in top-k results")
    total_keywords: int = Field(description="Total number of keywords to find")
    keyword_coverage: float = Field(description="Percentage of keywords found")


class AnswerEval(BaseModel):
    """LLM-as-a-judge evaluation of answer quality."""

    feedback: str = Field(
        description="Concise feedback

In [36]:
import importlib
import evaluation.eval as eval_module

eval_module = importlib.reload(eval_module)

print("fetch_context module:", eval_module.fetch_context.__module__)
print("answer_question module:", eval_module.answer_question.__module__)

fetch_context module: pro_implementation.proAnswer
answer_question module: pro_implementation.proAnswer


In [38]:
results = list(eval_module.evaluate_all_retrieval())

print("Number of results:", len(results))
print(results[:3])

Number of results: 150
[(TestQuestion(question='Who won the prestigious IIOTY award in 2023?', keywords=['Maxine', 'Thompson', 'IIOTY'], reference_answer='Maxine Thompson won the prestigious Insurellm Innovator of the Year (IIOTY) award in 2023.', category='direct_fact'), RetrievalEval(mrr=0.6666666666666666, ndcg=0.6105182098612075, keywords_found=2, total_keywords=3, keyword_coverage=66.66666666666666), 0.006666666666666667), (TestQuestion(question='When was Insurellm founded?', keywords=['2015', 'founded'], reference_answer='Insurellm was founded in 2015.', category='direct_fact'), RetrievalEval(mrr=1.0, ndcg=0.9723588025389089, keywords_found=2, total_keywords=2, keyword_coverage=100.0), 0.013333333333333334), (TestQuestion(question='Who founded Insurellm?', keywords=['Avery', 'Lancaster'], reference_answer='Avery Lancaster founded Insurellm in 2015.', category='direct_fact'), RetrievalEval(mrr=1.0, ndcg=1.0, keywords_found=2, total_keywords=2, keyword_coverage=100.0), 0.02)]


In [39]:
import pandas as pd

baseline_rows = []

for test, metrics, _ in results:
    baseline_rows.append({
        "category": test.category,
        "mrr": metrics.mrr,
        "ndcg": metrics.ndcg,
        "keyword_coverage": metrics.keyword_coverage
    })

baseline_df = pd.DataFrame(baseline_rows)

print("=== OVERALL BASELINE ===")
print(f"MRR:              {baseline_df['mrr'].mean():.4f}")
print(f"nDCG:             {baseline_df['ndcg'].mean():.4f}")
print(f"Keyword Coverage: {baseline_df['keyword_coverage'].mean():.2f}%")

print("\n=== BY CATEGORY ===")
print(
    baseline_df
    .groupby("category")[["mrr", "ndcg", "keyword_coverage"]]
    .mean()
    .sort_values("mrr")
)

=== OVERALL BASELINE ===
MRR:              0.8997
nDCG:             0.8733
Keyword Coverage: 96.29%

=== BY CATEGORY ===
                   mrr      ndcg  keyword_coverage
category                                          
holistic      0.735000  0.716758         80.000000
spanning      0.790208  0.795844         91.333333
numerical     0.879167  0.858374         95.000000
comparative   0.916667  0.920397        100.000000
direct_fact   0.928175  0.899610         98.095238
relationship  0.937500  0.927570        100.000000
temporal      0.975000  0.893498        100.000000


In [40]:
worst = []

for test, metrics, _ in results:
    worst.append({
        "question": test.question,
        "category": test.category,
        "mrr": metrics.mrr,
        "ndcg": metrics.ndcg,
        "keyword_coverage": metrics.keyword_coverage,
        "keywords": test.keywords
    })

worst_df = pd.DataFrame(worst)

worst_df = worst_df.sort_values(
    ["mrr", "ndcg", "keyword_coverage"]
)

worst_df.head(20)

,question,category,mrr,ndcg,keyword_coverage,keywords
58,What is the duration of the FastTrack Insuranc...,direct_fact,0.000000,0.000000,0.000000,"[18 months, FastTrack]"
101,Who is the technical lead for the product that...,spanning,0.035714,0.083333,25.000000,"[Robert, Chen, Homellm, $10,000]"
62,How many user licenses are included in Atlanti...,direct_fact,0.180556,0.396246,100.000000,"[35, Atlantic]"
133,How many user licenses are provided for onboar...,numerical,0.291667,0.527572,100.000000,"[15, Summit]"
117,How many core values does the company founded ...,spanning,0.361905,0.510295,80.000000,"[four, core values, Avery, Lancaster, founded]"
131,What is the monthly cost for Apex Reinsurance'...,numerical,0.500000,0.438424,50.000000,"[$10,000, Apex]"
147,What is the total number of active contracts a...,holistic,0.500000,0.477512,50.000000,"[contracts, total]"
144,What is the average monthly subscription cost ...,holistic,0.500000,0.485464,50.000000,"[Carllm, monthly cost]"
143,How many total employees work at Insurellm acr...,holistic,0.500000,0.495560,50.000000,"[employees, total]"
146,Which product has the fewest active contracts ...,holistic,0.500000,0.500000,50.000000,"[contracts, fewest]"


In [42]:
diagnostic_questions = [
    "What is the duration of the FastTrack Insurance product?",
    "How many user licenses are included in Atlantic Risk Solutions?",
    "Who is the technical lead for the product that costs $10,000?"
]

for question in diagnostic_questions:

    print("\n" + "=" * 80)
    print("QUESTION:", question)
    print("=" * 80)

    print("\n--- FINAL RERANKED RESULTS ---")

    final_chunks = fetch_context(question)

    for i, chunk in enumerate(final_chunks, 1):
        print(f"\n[{i}]")
        print(chunk.page_content[:500])
        print("SOURCE:", chunk.metadata.get("source"))


QUESTION: What is the duration of the FastTrack Insurance product?

--- FINAL RERANKED RESULTS ---

[1]
Terms of Coverage and Duration

Insurellm provides DriveSmart Insurance access to the Carllm platform, with the agreement lasting 36 months from March 20, 2025.

## Terms

1. **Coverage:** Insurellm agrees to provide DriveSmart Insurance with enterprise access to the Carllm platform, including white-label capabilities, unlimited user licenses, dedicated infrastructure, and full API access for their multi-state auto insurance operations.

2. **Duration:** This agreement is effective for a period 
SOURCE: /Users/tarkshya/Work/LLM Engineering/Week5_RAG/knowledge-base/contracts/Contract with DriveSmart Insurance for Carllm.md

[2]
Terms – Coverage and Duration

Defines the scope of services, including platform access, policy limits, infrastructure, and the contract's effective period.

## Terms

1. **Coverage:** Insurellm agrees to provide SafeHaven Property Insurance with enterprise ac

In [43]:
question = "What is the duration of the FastTrack Insurance product?"

rewritten = rewrite_query(question)

print("ORIGINAL:")
print(question)

print("\nREWRITTEN:")
print(rewritten)


print("\n" + "=" * 80)
print("ORIGINAL QUERY — TOP 20")
print("=" * 80)

original_chunks = fetch_context_unranked(question)

for i, chunk in enumerate(original_chunks, 1):
    print(f"\n[{i}] {chunk.metadata.get('source')}")
    print(chunk.page_content[:300])


print("\n" + "=" * 80)
print("REWRITTEN QUERY — TOP 20")
print("=" * 80)

rewritten_chunks = fetch_context_unranked(rewritten)

for i, chunk in enumerate(rewritten_chunks, 1):
    print(f"\n[{i}] {chunk.metadata.get('source')}")
    print(chunk.page_content[:300])

ORIGINAL:
What is the duration of the FastTrack Insurance product?

REWRITTEN:
FastTrack Insurance duration

ORIGINAL QUERY — TOP 20

[1] /Users/tarkshya/Work/LLM Engineering/Week5_RAG/knowledge-base/contracts/Contract with DriveSmart Insurance for Carllm.md
Terms of Coverage and Duration

Insurellm provides DriveSmart Insurance access to the Carllm platform, with the agreement lasting 36 months from March 20, 2025.

## Terms

1. **Coverage:** Insurellm agrees to provide DriveSmart Insurance with enterprise access to the Carllm platform, including white

[2] /Users/tarkshya/Work/LLM Engineering/Week5_RAG/knowledge-base/contracts/Contract with FastTrack Insurance Services for Claimllm.md
Contract Details and Parties Involved

This section details the basic information about the contract, including participating parties, date, and contract number.

**Contract Date:** May 10, 2025
**Contract Number:** CL-2025-0234

---

## Terms

1. **Parties Involved**: This contract is entered into 

[3

In [44]:
question = "What is the duration of the FastTrack Insurance product?"

original_chunks = fetch_context_unranked(question)

rewritten = rewrite_query(question)
rewritten_chunks = fetch_context_unranked(rewritten)

final_chunks = fetch_context(question)


def find_fasttrack(chunks):
    for rank, chunk in enumerate(chunks, 1):
        text = chunk.page_content.lower()
        source = chunk.metadata.get("source", "").lower()

        if "fasttrack" in text or "fasttrack" in source:
            return rank, chunk

    return None, None


original_rank, original_chunk = find_fasttrack(original_chunks)
rewritten_rank, rewritten_chunk = find_fasttrack(rewritten_chunks)
final_rank, final_chunk = find_fasttrack(final_chunks)


print("Rewritten query:", rewritten)

print("\nOriginal retrieval rank:", original_rank)
print("Rewritten retrieval rank:", rewritten_rank)
print("Final reranked rank:", final_rank)

Rewritten query: FastTrack Insurance duration

Original retrieval rank: 2
Rewritten retrieval rank: 1
Final reranked rank: None


In [45]:
from pro_implementation.proAnswer import (
    fetch_context_unranked,
    merge_chunks,
    rerank,
    rewrite_query,
)

question = "What is the duration of the FastTrack Insurance product?"

# Retrieve from both queries
original_chunks = fetch_context_unranked(question)

rewritten = rewrite_query(question)
rewritten_chunks = fetch_context_unranked(rewritten)

# Merge
merged_chunks = merge_chunks(original_chunks, rewritten_chunks)

# Rerank
reranked_chunks = rerank(question, merged_chunks)

print("Rewritten query:", rewritten)
print("Merged candidates:", len(merged_chunks))

print("\n" + "=" * 80)
print("RERANKED ORDER")
print("=" * 80)

for rank, chunk in enumerate(reranked_chunks, 1):
    source = chunk.metadata.get("source", "")

    marker = ""
    if "fasttrack" in source.lower() or "fasttrack" in chunk.page_content.lower():
        marker = "  <<< FASTTRACK"

    print(f"\n[{rank}] {source}{marker}")
    print(chunk.page_content[:250].replace("\n", " "))

Rewritten query: FastTrack Insurance duration
Merged candidates: 23

RERANKED ORDER

[1] /Users/tarkshya/Work/LLM Engineering/Week5_RAG/knowledge-base/contracts/Contract with DriveSmart Insurance for Carllm.md
Terms of Coverage and Duration  Insurellm provides DriveSmart Insurance access to the Carllm platform, with the agreement lasting 36 months from March 20, 2025.  ## Terms  1. **Coverage:** Insurellm agrees to provide DriveSmart Insurance with enterpr

[2] /Users/tarkshya/Work/LLM Engineering/Week5_RAG/knowledge-base/contracts/Contract with SafeHaven Property Insurance for Homellm.md
Terms – Coverage and Duration  Defines the scope of services, including platform access, policy limits, infrastructure, and the contract's effective period.  ## Terms  1. **Coverage:** Insurellm agrees to provide SafeHaven Property Insurance with ent

[3] /Users/tarkshya/Work/LLM Engineering/Week5_RAG/knowledge-base/contracts/Contract with Continental Commercial Group for Bizllm.md
Terms - Coverage an

In [46]:
# Find exactly where FastTrack ended up after reranking

fasttrack_rank = None
fasttrack_chunk = None

for rank, chunk in enumerate(reranked_chunks, 1):
    text = chunk.page_content.lower()
    source = chunk.metadata.get("source", "").lower()

    if "fasttrack" in text or "fasttrack" in source:
        fasttrack_rank = rank
        fasttrack_chunk = chunk
        break

print("FastTrack final rank:", fasttrack_rank)

if fasttrack_chunk:
    print("\nSOURCE:")
    print(fasttrack_chunk.metadata.get("source"))

    print("\nCONTENT:")
    print(fasttrack_chunk.page_content[:1500])

FastTrack final rank: 12

SOURCE:
/Users/tarkshya/Work/LLM Engineering/Week5_RAG/knowledge-base/contracts/Contract with FastTrack Insurance Services for Claimllm.md

CONTENT:
Contract Details and Parties Involved

This section details the basic information about the contract, including participating parties, date, and contract number.

**Contract Date:** May 10, 2025
**Contract Number:** CL-2025-0234

---

## Terms

1. **Parties Involved**: This contract is entered into between Insurellm, Inc. ("Provider") and FastTrack Insurance Services ("Client") on May 10, 2025.


In [47]:
print("=" * 80)
print("ALL FASTTRACK CANDIDATES")
print("=" * 80)

for rank, chunk in enumerate(merged_chunks, 1):
    text = chunk.page_content.lower()
    source = chunk.metadata.get("source", "").lower()

    if "fasttrack" in text or "fasttrack" in source:
        print(f"\nCANDIDATE #{rank}")
        print("SOURCE:", chunk.metadata.get("source"))
        print(chunk.page_content[:1200])

ALL FASTTRACK CANDIDATES

CANDIDATE #2
SOURCE: /Users/tarkshya/Work/LLM Engineering/Week5_RAG/knowledge-base/contracts/Contract with FastTrack Insurance Services for Claimllm.md
Contract Details and Parties Involved

This section details the basic information about the contract, including participating parties, date, and contract number.

**Contract Date:** May 10, 2025
**Contract Number:** CL-2025-0234

---

## Terms

1. **Parties Involved**: This contract is entered into between Insurellm, Inc. ("Provider") and FastTrack Insurance Services ("Client") on May 10, 2025.

CANDIDATE #7
SOURCE: /Users/tarkshya/Work/LLM Engineering/Week5_RAG/knowledge-base/contracts/Contract with FastTrack Insurance Services for Claimllm.md
License and Payment Terms

This section covers the license grant to the client and the payment structure, including monthly fees and total contract value.

2. **License Grant**: Insurellm grants FastTrack Insurance Services a non-exclusive, non-transferable license to us

In [48]:
fasttrack_source = "/Users/tarkshya/Work/LLM Engineering/Week5_RAG/knowledge-base/contracts/Contract with FastTrack Insurance Services for Claimllm.md"

fasttrack_data = baseline.collection.get(
    where={"source": fasttrack_source},
    include=["documents", "metadatas"]
)

print("FastTrack chunks:", len(fasttrack_data["documents"]))

for i, (doc, meta) in enumerate(
    zip(fasttrack_data["documents"], fasttrack_data["metadatas"]), 1
):
    print("\n" + "=" * 80)
    print(f"CHUNK {i}")
    print("=" * 80)
    print(doc[:1500])

FastTrack chunks: 8

CHUNK 1
Contract Details and Parties Involved

This section details the basic information about the contract, including participating parties, date, and contract number.

**Contract Date:** May 10, 2025
**Contract Number:** CL-2025-0234

---

## Terms

1. **Parties Involved**: This contract is entered into between Insurellm, Inc. ("Provider") and FastTrack Insurance Services ("Client") on May 10, 2025.

CHUNK 2
License and Payment Terms

This section covers the license grant to the client and the payment structure, including monthly fees and total contract value.

2. **License Grant**: Insurellm grants FastTrack Insurance Services a non-exclusive, non-transferable license to use the Claimllm Advanced Tier platform.

3. **Payment Terms**: FastTrack Insurance Services agrees to make monthly payments of $9,500 for the duration of this 18-month contract, totaling $171,000. Payments due on the 1st of each month.

CHUNK 3
Claims Volume and Confidentiality

Details the cl

In [49]:
for i, doc in enumerate(fasttrack_data["documents"], 1):
    if any(word in doc.lower() for word in ["duration", "effective", "months", "term"]):
        print("\n" + "=" * 80)
        print(f"CHUNK {i}")
        print("=" * 80)
        print(doc)


CHUNK 1
Contract Details and Parties Involved

This section details the basic information about the contract, including participating parties, date, and contract number.

**Contract Date:** May 10, 2025
**Contract Number:** CL-2025-0234

---

## Terms

1. **Parties Involved**: This contract is entered into between Insurellm, Inc. ("Provider") and FastTrack Insurance Services ("Client") on May 10, 2025.

CHUNK 2
License and Payment Terms

This section covers the license grant to the client and the payment structure, including monthly fees and total contract value.

2. **License Grant**: Insurellm grants FastTrack Insurance Services a non-exclusive, non-transferable license to use the Claimllm Advanced Tier platform.

3. **Payment Terms**: FastTrack Insurance Services agrees to make monthly payments of $9,500 for the duration of this 18-month contract, totaling $171,000. Payments due on the 1st of each month.

CHUNK 3
Claims Volume and Confidentiality

Details the claims volume support 

In [50]:
for i, doc in enumerate(fasttrack_data["documents"], 1):
    text = doc.lower()

    if "duration" in text or "effective period" in text or "18 months" in text:
        print("\n" + "=" * 80)
        print(f"CHUNK {i}")
        print("=" * 80)

        # Print only the relevant surrounding text
        for keyword in ["duration", "effective period", "18 months"]:
            pos = text.find(keyword)

            if pos != -1:
                start = max(0, pos - 500)
                end = min(len(doc), pos + 1000)

                print(doc[start:end])
                print("\n---")


CHUNK 2
License and Payment Terms

This section covers the license grant to the client and the payment structure, including monthly fees and total contract value.

2. **License Grant**: Insurellm grants FastTrack Insurance Services a non-exclusive, non-transferable license to use the Claimllm Advanced Tier platform.

3. **Payment Terms**: FastTrack Insurance Services agrees to make monthly payments of $9,500 for the duration of this 18-month contract, totaling $171,000. Payments due on the 1st of each month.

---

CHUNK 3
Claims Volume and Confidentiality

Details the claims volume support and confidentiality commitments by both parties, with confidentiality lasting for 5 years post-contract.

4. **Claims Volume**: Advanced Tier supports up to 25,000 claims annually. FastTrack projects 22,000 claims in year 1.

5. **Confidentiality**: Both parties shall maintain confidentiality regarding each other's proprietary information for the contract duration plus 5 years post-termination.

---

In [51]:
question = "What is the duration of the FastTrack Insurance product?"

chunks = fetch_context_unranked(question)

for rank, chunk in enumerate(chunks, 1):
    if "18-month" in chunk.page_content.lower():
        print("FOUND!")
        print("Rank:", rank)
        print(chunk.page_content)

FOUND!
Rank: 7
License and Payment Terms

This section covers the license grant to the client and the payment structure, including monthly fees and total contract value.

2. **License Grant**: Insurellm grants FastTrack Insurance Services a non-exclusive, non-transferable license to use the Claimllm Advanced Tier platform.

3. **Payment Terms**: FastTrack Insurance Services agrees to make monthly payments of $9,500 for the duration of this 18-month contract, totaling $171,000. Payments due on the 1st of each month.


In [53]:
from openai import OpenAI
openai = OpenAI()

query = openai.embeddings.create(
    model=baseline.embedding_model,
    input=[question]
).data[0].embedding

for k in [20, 40, 60, 100]:

    results_k = baseline.collection.query(
        query_embeddings=[query],
        n_results=k
    )

    found = None

    for rank, doc in enumerate(results_k["documents"][0], 1):
        if "18-month" in doc.lower():
            found = rank
            break

    print(f"K={k:3} → 18-month chunk rank: {found}")

K= 20 → 18-month chunk rank: 7
K= 40 → 18-month chunk rank: 7
K= 60 → 18-month chunk rank: 7
K=100 → 18-month chunk rank: 7


In [54]:
gold_rank = None

for rank, chunk in enumerate(reranked_chunks, 1):
    if "18-month contract" in chunk.page_content.lower():
        gold_rank = rank

        print("Gold duration chunk final rank:", rank)
        print("\nSOURCE:")
        print(chunk.metadata.get("source"))

        print("\nCONTENT:")
        print(chunk.page_content)

        break

print("\nGold rank:", gold_rank)

Gold duration chunk final rank: 14

SOURCE:
/Users/tarkshya/Work/LLM Engineering/Week5_RAG/knowledge-base/contracts/Contract with FastTrack Insurance Services for Claimllm.md

CONTENT:
License and Payment Terms

This section covers the license grant to the client and the payment structure, including monthly fees and total contract value.

2. **License Grant**: Insurellm grants FastTrack Insurance Services a non-exclusive, non-transferable license to use the Claimllm Advanced Tier platform.

3. **Payment Terms**: FastTrack Insurance Services agrees to make monthly payments of $9,500 for the duration of this 18-month contract, totaling $171,000. Payments due on the 1st of each month.

Gold rank: 14
